# nb00 — Extract Medi-Cal Managed Care Enrollment Data

<hr style="border: 3px solid black;">

**Purpose:** Pull the latest Medi-Cal Managed Care Enrollment Report from the CalHHS Open Data Portal (CKAN API), save a raw copy, and run a first profile.

**Why the API instead of the download button:** the CSV filename changes every monthly refresh (e.g. `open-data-portal-managed-care-enrollment-count-may-2026.csv`). The CKAN `package_show` endpoint always resolves to the current file, so this notebook never breaks when DHCS publishes a new month.

**Dataset:** https://data.chhs.ca.gov/dataset/medi-cal-managed-care-enrollment-report

**Pipeline position:** nb00 (extract) → nb01 (clean/prepare for Tableau, SQL, pandas parity work)

## 1. Setup and configuration

In [1]:
import json
from datetime import date
from pathlib import Path

import pandas as pd
import requests

# CKAN API configuration
CKAN_BASE = 'https://data.chhs.ca.gov/api/3/action'
DATASET_SLUG = 'medi-cal-managed-care-enrollment-report'

# Output locations (notebook lives in notebooks/, data lives in ../data/)
DATA_DIR = Path('..') / 'data'
RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('Raw data folder:', RAW_DIR.resolve())

Raw data folder: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/tableau_medi_cal/data/raw


## 2. Resolve the current CSV resource via the CKAN API

`package_show` returns dataset metadata including every attached resource. We grab the CSV resource, its live download URL, and its last modified date.

In [2]:
resp = requests.get(f'{CKAN_BASE}/package_show', params={'id': DATASET_SLUG}, timeout=60)
resp.raise_for_status()
pkg = resp.json()['result']

csv_resources = [r for r in pkg['resources'] if r.get('format', '').upper() == 'CSV']
resource = csv_resources[0]

print('Dataset title :', pkg['title'])
print('Resource name :', resource['name'])
print('Last modified :', resource.get('last_modified'))
print('Download URL  :', resource['url'])

Dataset title : Medi-Cal Managed Care Enrollment Report
Resource name : Medi-Cal Managed Care Enrollment Report
Last modified : 2026-07-01T22:03:04.335947
Download URL  : https://data.chhs.ca.gov/dataset/c6ccef54-e7a9-4ebd-b79a-850b72c4dd8c/resource/95358a7a-2c9d-41c6-a0e0-405a7e5c5f18/download/open-data-portal-managed-care-enrollment-count-june-2026.csv


## 3. Download the CSV and save a date stamped raw copy

Raw files are immutable inputs: never edit them. Cleaning happens downstream in nb01.

In [3]:
download = requests.get(resource['url'], timeout=300)
download.raise_for_status()

raw_path = RAW_DIR / f"medi_cal_mc_enrollment_raw_{date.today().isoformat()}.csv"
raw_path.write_bytes(download.content)

size_mb = raw_path.stat().st_size / 1_048_576
print(f'Saved {raw_path.name} ({size_mb:.1f} MB)')

Saved medi_cal_mc_enrollment_raw_2026-07-05.csv (2.5 MB)


## 4. First profile

Confirm the grain (one row per month × county × plan), the date range, and that LA County plans (including L.A. Care) are present.

In [4]:
df = pd.read_csv(raw_path)

print('Shape:', df.shape)
print()
print(df.dtypes)
df.head()

Shape: (31178, 8)

Enrollment Month                              object
Plan Type                                     object
County                                        object
Plan Name                                     object
 Count of Enrollees                           object
Count of Enrollees Annotation Code           float64
Count of Enrollees Annotation Description     object
Unnamed: 7                                   float64
dtype: object


,Enrollment Month,Plan Type,County,Plan Name,Count of Enrollees,Count of Enrollees Annotation Code,Count of Enrollees Annotation Description,Unnamed: 7
0,2007-01,COHS,Orange,CalOPTIMA / Orange,"293,311",NaN,NaN,NaN
1,2007-01,COHS,Monterey,Central California Alliance for Health/Monterey,"53,819",NaN,NaN,NaN
2,2007-01,COHS,Santa Cruz,Central California Alliance for Health/Santa Cruz,"30,653",NaN,NaN,NaN
3,2007-01,COHS,San Mateo,Hlth Plan of San Mateo,"49,791",NaN,NaN,NaN
4,2007-01,COHS,Yolo,Partnership HealthPlan of CA/Yolo,"23,515",NaN,NaN,NaN


In [5]:
# Column inventory: unique counts and example values
for col in df.columns:
    nunique = df[col].nunique()
    examples = df[col].dropna().unique()[:5]
    print(f'{col}: {nunique} unique — e.g. {list(examples)}')

Enrollment Month: 234 unique — e.g. ['2007-01', '2007-02', '2007-03', '2007-04', '2007-05']
Plan Type: 20 unique — e.g. ['COHS', 'Dental', 'Geographic Managed Care', 'PACE', 'Prepaid Health Plan']
County: 59 unique — e.g. ['Orange', 'Monterey', 'Santa Cruz', 'San Mateo', 'Yolo']
Plan Name: 477 unique — e.g. ['CalOPTIMA / Orange', 'Central California Alliance for Health/Monterey', 'Central California Alliance for Health/Santa Cruz', 'Hlth Plan of San Mateo', 'Partnership HealthPlan of CA/Yolo']
 Count of Enrollees : 21160 unique — e.g. [' 293,311 ', ' 53,819 ', ' 30,653 ', ' 49,791 ', ' 23,515 ']
Count of Enrollees Annotation Code: 2 unique — e.g. [1.0, 2.0]
Count of Enrollees Annotation Description: 2 unique — e.g. ['Cell suppressed for small numbers', 'Cell suppressed for complementary cell']
Unnamed: 7: 0 unique — e.g. []


In [6]:
# Date range covered (adjust column name after inspecting the inventory above)
date_col = [c for c in df.columns if 'month' in c.lower() or 'date' in c.lower()][0]
print('Date column:', date_col)
print('Range:', df[date_col].min(), 'to', df[date_col].max())

Date column: Enrollment Month
Range: 2007-01 to 2026-06


In [7]:
# Los Angeles County check: which plans report enrollment there?
county_col = [c for c in df.columns if 'county' in c.lower()][0]
la = df[df[county_col].astype(str).str.contains('Los Angeles', case=False, na=False)]
print('LA County rows:', len(la))
plan_col = [c for c in df.columns if 'plan' in c.lower() and 'type' not in c.lower()][0]
print('Plans in LA County:')
print(la[plan_col].value_counts())

LA County rows: 3495
Plans in LA County:
Plan Name
Access Dental Plan/LA                                  196
Health Net of California                               196
Liberty Dental Plan of CA (LA)                         196
SCAN (Nurs hm cert) / LA                               188
Health Net / LA                                        188
AltaMed Hlth Sr. Buena Care/LA                         188
SCAN Hlth Plan / LA                                    188
LA CARE                                                188
AIDS HlthCare Found/ LA                                187
Brandman Ctrs for Senior Care                          115
Molina Healthcare Los Angeles CCI                      101
Health Net Los Angeles CCI                              96
LA CARE CCI                                             96
SCAN Health Plan/Los Angeles                            92
Western Dental Svcs/LA                                  78
Safeguard Dental Inc.                                   78
Care1

## 5. Extraction log

Record what was pulled so the blog post and parity work reference an exact vintage.

In [8]:
log = {
    'extracted_on': date.today().isoformat(),
    'dataset': pkg['title'],
    'resource_name': resource['name'],
    'resource_last_modified': resource.get('last_modified'),
    'source_url': resource['url'],
    'raw_file': raw_path.name,
    'rows': int(df.shape[0]),
    'columns': list(df.columns),
}

log_path = DATA_DIR / 'extraction_log.json'
log_path.write_text(json.dumps(log, indent=2))
print(json.dumps(log, indent=2))

{
  "extracted_on": "2026-07-05",
  "dataset": "Medi-Cal Managed Care Enrollment Report",
  "resource_name": "Medi-Cal Managed Care Enrollment Report",
  "resource_last_modified": "2026-07-01T22:03:04.335947",
  "source_url": "https://data.chhs.ca.gov/dataset/c6ccef54-e7a9-4ebd-b79a-850b72c4dd8c/resource/95358a7a-2c9d-41c6-a0e0-405a7e5c5f18/download/open-data-portal-managed-care-enrollment-count-june-2026.csv",
  "raw_file": "medi_cal_mc_enrollment_raw_2026-07-05.csv",
  "rows": 31178,
  "columns": [
    "Enrollment Month",
    "Plan Type",
    "County",
    "Plan Name",
    " Count of Enrollees ",
    "Count of Enrollees Annotation Code",
    "Count of Enrollees Annotation Description",
    "Unnamed: 7"
  ]
}


---

**Next:** run nb01 to clean plan/county fields, parse the month column to a real date, and export the Tableau ready CSV that SQL (DuckDB) and pandas will also consume.